In [6]:
import pandas as pd
import os
import requests
import sqlite3
from dotenv import load_dotenv
import json

In [3]:
load_dotenv()
TOKEN = os.getenv("TOKEN")
CITY_NAME = ["Dakar", "Libreville", "Abidjan", "Lagos", "Accra"]


# = f"https://api.openweathermap.org/data/2.5/weather?q={CITY_NAME}&appid={TOKEN}"

In [10]:
print(f"API Token: {TOKEN}")

API Token: 37e117d1a5ee7d5d0aaf0243e03f4129


In [8]:
ville = "Dakar"

def ingest_data():
    url=f"https://api.openweathermap.org/data/2.5/weather?q={ville}&appid={TOKEN}"
    response = requests.get(url)
    data = response.json()
    return data

def transform_data(data):
    dic = {
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "pressure": data["main"]["pressure"],
        "wind_speed": data["wind"]["speed"],
        "weather_description": data["weather"][0]["description"]
    }
    df = pd.DataFrame([dic])
    return df

if __name__ == "__main__":
    data = ingest_data()
    print(json.dumps(data, indent=4))
    df = transform_data(data)
    #print(df)

    
    



{
    "coord": {
        "lon": -17.4441,
        "lat": 14.6937
    },
    "weather": [
        {
            "id": 804,
            "main": "Clouds",
            "description": "overcast clouds",
            "icon": "04n"
        }
    ],
    "base": "stations",
    "main": {
        "temp": 301.07,
        "feels_like": 305.82,
        "temp_min": 301.07,
        "temp_max": 301.07,
        "pressure": 1013,
        "humidity": 85,
        "sea_level": 1013,
        "grnd_level": 1013
    },
    "visibility": 10000,
    "wind": {
        "speed": 6.57,
        "deg": 334,
        "gust": 6.49
    },
    "clouds": {
        "all": 88
    },
    "dt": 1785203397,
    "sys": {
        "type": 2,
        "id": 2103129,
        "country": "SN",
        "sunrise": 1785221508,
        "sunset": 1785267631
    },
    "timezone": 0,
    "id": 2253354,
    "name": "Dakar",
    "cod": 200
}


In [ ]:
def extract():

    weather_data = []

    for city in CITY_NAME:
        city_url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={TOKEN}"
        response = requests.get(city_url)
        weather_data.append(response.json())
    return weather_data


def transform(data):
    #selections des variables
    weather = []

    for city_data in data:
        city_weather = {
            "city": city_data["name"],
            "humidity": city_data["main"]["humidity"],
            "temperature": city_data["main"]["temp"],
            "pressure": city_data["main"]["pressure"],
            "wind_speed": city_data["wind"]["speed"]
        }
        weather.append(city_weather)
    df = pd.DataFrame(weather)
    df.to_csv("transformed_weather_data.csv", index=False)
    return df

def load(db_path, df):
    try:
        df = pd.read_csv("transformed_weather_data.csv")
        #ouvrir une connexion à la base de données SQLite
        conn = sqlite3.connect(db_path)
        #permettant d'executer des commandes SQL
        print(f"Connexion à la base {db_path} établie")
        #cette ligne prend le dataframe et le converti en table sql
        df.to_sql("weather", conn, if_exists="replace", index=False)
        print("Données insérées dans la table weather")

        #valide la transaction et ferme la connexion
        conn.commit()
        conn.close()

    except sqlite3.Error as e:
        print(f"Erreur lors de l'insertion des données dans la base de données: {e}")


    



if __name__ == "__main__":
    print("=============================")
    print("Extraction des donnees")
    print("=============================")
    data = extract()
    print("=============================")
    print("Transformation des donnees")
    print("=============================")
    df = transform(data)

    print("=============================")
    print("Chargement des donnees dans la base de donnees")
    print("=============================")
    loader = load("weather_data.db", df)

Extraction des donnees
Transformation des donnees
Chargement des donnees dans la base de donnees
Connexion à la base weather_data.db établie
Données insérées dans la table weather
